In [3]:
import pyspark 

sc = pyspark.SparkContext.getOrCreate()

d7_rdd = sc.textFile("/Users/mary/Desktop/Mary/MSDS_fall_2_26/694/GRP/startup-pharmacy/data/D7_DHCS_2024.csv")

d7_rdd.take(5)

['Month of Eligibility,Zip Code,Sex,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data',
 '2020-01,        .,Female,"54,989",,,F,11JAN2021',
 '2020-01,        .,Male,"77,134",,,F,11JAN2021',
 '2020-01,        .,Total,"132,123",,,F,11JAN2021',
 '2020-01,90001,Female,"21,985",,,F,11JAN2021']

In [5]:
# filter 2024, use total count (both sex), count he records and display 

d7_2024 = d7_rdd.filter(lambda x: x.startswith("2024-"))
print("2024 total records: ", d7_2024.count())
d7_2024.take(5)

2024 total records:  93333


['2024-01,        .,Female,"80,542",,,F,07JAN2025',
 '2024-01,        .,Male,"113,138",,,F,07JAN2025',
 '2024-01,        .,Total,"193,680",,,F,07JAN2025',
 '2024-01,90001,Female,"26,853",,,F,07JAN2025',
 '2024-01,90001,Male,"22,490",,,F,07JAN2025']

In [7]:
# filter total count (both sex), remove empty zip code, get the eligibility count then convert. 
d7_total = d7_2024.filter(
    lambda x: x.split(",")[2] == "Total"
)
print("Total eligibility records:", d7_total.count())
d7_total.take(5)

Total eligibility records: 31119


['2024-01,        .,Total,"193,680",,,F,07JAN2025',
 '2024-01,90001,Total,"49,343",,,F,07JAN2025',
 '2024-01,90002,Total,"36,403",,,F,07JAN2025',
 '2024-01,90003,Total,"53,774",,,F,07JAN2025',
 '2024-01,90004,Total,"27,185",,,F,07JAN2025']

In [ ]:
# filter zipcode and get valid 5 digit zipcode 
d7_clean_zipcode = d7_total.filter(
    lambda x: x.split(",")[1].strip()
!= ".")

print(d7_clean_zipcode.count())
d7_clean_zipcode.take(5)

# Removed 12 records due to missing zip code. 

31107


['2024-01,90001,Total,"49,343",,,F,07JAN2025',
 '2024-01,90002,Total,"36,403",,,F,07JAN2025',
 '2024-01,90003,Total,"53,774",,,F,07JAN2025',
 '2024-01,90004,Total,"27,185",,,F,07JAN2025',
 '2024-01,90005,Total,"19,242",,,F,07JAN2025']

In [24]:
d7_eligibility = d7_clean_zipcode.map(
    lambda x: (
        x.split('"')[0],
        x.split(",")[1].strip(),
        int(x.split('"')[1].replace(",", ""))
        if '"' in x
        else (int(x.split(",")[3]) if x.split(",")[3] != "" else None) 
    )
)
d7_eligibility.take(5)

[('2024-01,90001,Total,', '90001', 49343),
 ('2024-01,90002,Total,', '90002', 36403),
 ('2024-01,90003,Total,', '90003', 53774),
 ('2024-01,90004,Total,', '90004', 27185),
 ('2024-01,90005,Total,', '90005', 19242)]

In [26]:
# Check invalid Zipcode 
d7_invalid_zip = d7_eligibility.filter(
    lambda x: len(x[1]) != 5 or not x[1].isdigit()
)
print("Invalid zip code: ", d7_invalid_zip.count())

d7_invalid_zip.take(12)

Invalid zip code:  12


[('2024-01,STATEWIDE,Total,', 'STATEWIDE', 15158900),
 ('2024-02,STATEWIDE,Total,', 'STATEWIDE', 15145633),
 ('2024-03,STATEWIDE,Total,', 'STATEWIDE', 15108208),
 ('2024-04,STATEWIDE,Total,', 'STATEWIDE', 15053072),
 ('2024-05,STATEWIDE,Total,', 'STATEWIDE', 14997322),
 ('2024-06,STATEWIDE,Total,', 'STATEWIDE', 14913748),
 ('2024-07,STATEWIDE,Total,', 'STATEWIDE', 14909860),
 ('2024-08,STATEWIDE,Total,', 'STATEWIDE', 14919922),
 ('2024-09,STATEWIDE,Total,', 'STATEWIDE', 14901642),
 ('2024-10,STATEWIDE,Total,', 'STATEWIDE', 14904024),
 ('2024-11,STATEWIDE,Total,', 'STATEWIDE', 14892769),
 ('2024-12,STATEWIDE,Total,', 'STATEWIDE', 14895996)]

In [27]:
#Invalid zip is CA eligibility count total. keep it but separate from the rdd. 

d7_statewide_count = d7_eligibility.filter(
    lambda x: x[1] == "STATEWIDE"
)

d7_zip = d7_eligibility.filter(
    lambda x: x[1] != "STATEWIDE"
)

print("Statewide records: should be 12 ", d7_statewide_count.count())
print("ZIP records ", d7_zip.count())

Statewide records: should be 12  12
ZIP records  31095


In [29]:
# Missing eligibility check 
d7_missing = d7_zip.filter(
    lambda x: x[2] == None
)

print("Missing eligibility records:", d7_missing.count())
print("**Limitation** This missing records is about 24.4% total : 31095")

Missing eligibility records: 7594
**Limitation** This missing records is about 24.4% total : 31095


In [30]:
# Keep the records with available eligibility counts 
d7_available_counts = d7_zip.filter(lambda x: x[2] != None)
print("available records:", d7_available_counts.count())

available records: 23501


In [ ]:
# Group eligibility by zipcode 

